# Desplegar la app web del mapa de validación (Databricks Apps)
Este notebook se ejecuta en su computadora, conectado a Databricks con el perfil de la CLI. Hace tres cosas:
1. **Sube el código de la app** (`validacion_mapa/app/` y la plantilla `validacion_mapa/plantilla.html`) a la carpeta `app_mapa` del workspace.
2. **Crea (o actualiza) la app** `mapa-validacion` con permiso de lectura sobre el Volume `web`.
3. **Despliega** el código y muestra la URL.

Los datos los prepara la tarea `mapa_validacion` del job (notebook `12_mapa_validacion`); cada vez que el job corre, la app muestra los datos nuevos sin volver a desplegarla.

En la Free Edition la app se apaga sola a las 24 horas: se vuelve a encender con la última sección o desde **Compute → Apps**. La key de Google Maps no se sube: la app usa los fondos de Esri, OpenStreetMap y Carto.

## 1. Conexión a Databricks

In [ ]:
# Librerías y conexión al workspace de Databricks.
from pathlib import Path

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service import apps, catalog
from databricks.sdk.service.workspace import ImportFormat

PERFIL = "22200144"
w = WorkspaceClient(profile=PERFIL)
usuario = w.current_user.me().user_name
print("Conectado como", usuario)

## 2. Configuración

In [ ]:
# Nombre de la app, rutas y archivos que se suben.
CATALOGO, ESQUEMA, VOLUMEN = "grupo06proyectofinal", "sismos", "web"
NOMBRE_APP = "mapa-validacion"
RUTA_APP = f"/Users/{usuario}/sismos_lakehouse/app_mapa"
ARCHIVOS_APP = {
    "app.py": Path("validacion_mapa/app/app.py"),
    "app.yaml": Path("validacion_mapa/app/app.yaml"),
    "requirements.txt": Path("validacion_mapa/app/requirements.txt"),
    "plantilla.html": Path("validacion_mapa/plantilla.html"),
}
ARCHIVO_KEY_GOOGLE = Path("validacion_mapa/google_maps_key.txt")

## 3. Funciones

In [ ]:
# Crea el Volume web si no existe.
def asegurar_volumen():
    try:
        w.volumes.read(f"{CATALOGO}.{ESQUEMA}.{VOLUMEN}")
    except NotFound:
        w.volumes.create(catalog_name=CATALOGO, schema_name=ESQUEMA, name=VOLUMEN, volume_type=catalog.VolumeType.MANAGED)
    print("Volume de la app:", f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN}")


# Sube los archivos de la app al workspace, sin secretos.
def subir_codigo():
    key = ARCHIVO_KEY_GOOGLE.read_text().strip() if ARCHIVO_KEY_GOOGLE.exists() else ""
    w.workspace.mkdirs(RUTA_APP)
    for nombre, local in ARCHIVOS_APP.items():
        contenido = local.read_bytes()
        assert not key or key.encode() not in contenido, f"{local} contiene la key de Google: no se sube"
        w.workspace.upload(f"{RUTA_APP}/{nombre}", contenido, format=ImportFormat.RAW, overwrite=True)
        print(f"  {local} -> {RUTA_APP}/{nombre} ({len(contenido) / 1e3:.0f} KB)")


# Crea la app o actualiza su configuración, y da permisos a su service principal.
def crear_o_actualizar_app():
    volumen_web = apps.AppResource(
        name="volumen-web", description="Datos del mapa (solo lectura)",
        uc_securable=apps.AppResourceUcSecurable(
            securable_full_name=f"{CATALOGO}.{ESQUEMA}.{VOLUMEN}",
            securable_type=apps.AppResourceUcSecurableUcSecurableType.VOLUME,
            permission=apps.AppResourceUcSecurableUcSecurablePermission.READ_VOLUME))
    config = apps.App(name=NOMBRE_APP, resources=[volumen_web],
                      description="Mapa de validación espacial de los 6 distritos de estudio (lakehouse de sismos)")
    try:
        w.apps.get(NOMBRE_APP)
        app = w.apps.update(name=NOMBRE_APP, app=config)
    except NotFound:
        app = w.apps.create_and_wait(app=config)
    sp = app.service_principal_client_id
    for tipo, nombre, privilegio in [("catalog", CATALOGO, catalog.Privilege.USE_CATALOG),
                                     ("schema", f"{CATALOGO}.{ESQUEMA}", catalog.Privilege.USE_SCHEMA)]:
        w.grants.update(securable_type=tipo, full_name=nombre, changes=[catalog.PermissionsChange(principal=sp, add=[privilegio])])
    print("App:", app.name, "| service principal:", app.service_principal_name, sp)
    return app


# Enciende la app si está apagada.
def encender():
    app = w.apps.get(NOMBRE_APP)
    if app.compute_status is None or app.compute_status.state != apps.ComputeState.ACTIVE:
        print("Encendiendo el cómputo de la app...")
        w.apps.start_and_wait(NOMBRE_APP)


# Lanza un despliegue de la app y espera a que termine.
def desplegar():
    encender()
    despliegue = w.apps.deploy_and_wait(app_name=NOMBRE_APP,
                                        app_deployment=apps.AppDeployment(source_code_path=f"/Workspace{RUTA_APP}"))
    print("Despliegue:", despliegue.status.state.value, "|", despliegue.status.message)


# Muestra el estado actual: sismos que llegaron o situación de la app.
def estado():
    app = w.apps.get(NOMBRE_APP)
    print("Cómputo:", app.compute_status.state.value if app.compute_status else None,
          "| app:", app.app_status.state.value if app.app_status else None,
          "|", app.app_status.message if app.app_status else "")
    print("URL:", app.url)

## 4. Volume y código de la app
Antes de subirlos se comprueba que ningún archivo lleve la key de Google.

In [ ]:
# Crea el Volume web y sube el código de la app.
asegurar_volumen()
subir_codigo()

## 5. Crear o actualizar la app
La app lee el Volume `web` con su service principal (permiso `READ VOLUME`, más `USE CATALOG` y `USE SCHEMA`).

In [ ]:
# Crea la app o actualiza su configuración.
app = crear_o_actualizar_app()

## 6. Desplegar

In [ ]:
# Despliega la app y muestra su estado.
desplegar()
estado()

## 7. Volver a encender la app
En la Free Edition se apaga sola a las 24 horas.

In [ ]:
# Enciende la app y muestra su estado.
encender()
estado()